# Cognitive Policies

When making important decisions, humans naturally gather information first (Acquiring), mentally rehearse plans before acting (Rehearsal), and assess whether information is reliable (Reflection). Cognitive Policies bring this same deliberation to your agents — optional multi-round thinking within a single OTC cycle.

In this tutorial, we'll build an **investment analysis agent** that uses all three policies: it gathers skill details before deciding, rehearses its investment recommendation, and reflects on data consistency.

## Initialize

First, let's set up the LLM and define the tools our investment analyst will use.

In [ ]:
import os
from bridgic.model import OpenAILlm

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = LLM(model="openai/gpt-4o-mini")

In [ ]:
from bridgic.core import tool


@tool(description="Fetch stock price data for a ticker symbol")
async def get_stock_price(ticker: str) -> str:
    prices = {"AAPL": "$178.50", "GOOGL": "$141.20", "MSFT": "$415.30", "TSLA": "$245.80"}
    return f"{ticker}: {prices.get(ticker, 'Unknown ticker')}"


@tool(description="Get financial metrics for a company")
async def get_financials(company: str) -> str:
    return f"{company} - P/E: 28.5, Revenue Growth: 12%, Debt/Equity: 0.45"


@tool(description="Analyze market sentiment for a stock")
async def analyze_sentiment(ticker: str) -> str:
    return f"Sentiment for {ticker}: 65% bullish, 25% neutral, 10% bearish"


@tool(description="Generate an investment recommendation report")
async def generate_recommendation(ticker: str, analysis: str) -> str:
    return f"Investment Report for {ticker}: {analysis}"

We have four tools:

- **`get_stock_price`** — fetches the current stock price for a given ticker symbol.
- **`get_financials`** — retrieves key financial metrics (P/E ratio, revenue growth, debt/equity).
- **`analyze_sentiment`** — provides a market sentiment breakdown (bullish/neutral/bearish percentages).
- **`generate_recommendation`** — produces a final investment recommendation report.

## Acquiring — Gathering Details on Demand

Acquiring is the **built-in, always-active** policy. When the agent's context uses `LayeredExposure` (like skills or history), the LLM first sees summaries. If it needs more detail, it fills the `details` field in its response, and the framework reveals the requested information before the next thinking round.

This means the LLM can work with compact summaries most of the time, only expanding into full detail when it determines that extra information is needed — keeping token usage efficient without sacrificing access to depth.

The Acquiring policy fires at most once per OTC cycle, then closes.

In [ ]:
from bridgic.amphibious import (
    AmphibiousAutoma, CognitiveContext, CognitiveWorker, think_unit
)


class InvestmentAnalyst(AmphibiousAutoma[CognitiveContext]):
    analyst = think_unit(
        CognitiveWorker.inline(
            "Analyze the investment opportunity. "
            "Gather stock prices, financial data, and sentiment before making a recommendation."
        ),
        max_attempts=8,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.analyst


agent = InvestmentAnalyst(llm=llm)
result = await agent.arun(
    goal="Analyze AAPL stock and provide an investment recommendation",
    tools=[get_stock_price, get_financials, analyze_sentiment, generate_recommendation],
)
print(result)

The Acquiring policy fires automatically when the LLM requests details from `LayeredExposure` fields (skills, cognitive_history). It fires at most once per OTC cycle, then closes. You do not need to explicitly enable it — it is always available as part of the framework's core behavior.

## Rehearsal — Mentally Simulating Before Acting

Rehearsal lets the agent "mentally simulate" its planned action before committing. This is especially valuable for high-risk operations — the agent can catch potential issues before they happen.

With rehearsal enabled, before committing to a tool call, the LLM can fill a `rehearsal` field to simulate what would happen. The framework processes this simulation round and then asks the LLM to make its final decision. Think of it as the agent asking itself: *"If I do this, what will happen?"*

In [ ]:
class CautiousAnalyst(AmphibiousAutoma[CognitiveContext]):
    analyst = think_unit(
        CognitiveWorker.inline(
            "Analyze the stock and generate an investment recommendation. "
            "This is a high-stakes decision — think carefully before committing.",
            enable_rehearsal=True,  # Enable mental rehearsal
        ),
        max_attempts=8,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.analyst


agent = CautiousAnalyst(llm=llm)
result = await agent.arun(
    goal="Analyze TSLA stock — the client has a $100K budget and low risk tolerance",
    tools=[get_stock_price, get_financials, analyze_sentiment, generate_recommendation],
)
print(result)

With rehearsal enabled, the agent can mentally simulate its planned recommendation before committing. For a high-stakes scenario like this — a large budget with low risk tolerance — the rehearsal step gives the LLM an opportunity to reconsider its approach if the simulation reveals potential problems.

Rehearsal is ideal for situations where mistakes are costly: financial decisions, irreversible actions, or operations with significant side effects.

## Reflection — Assessing Information Quality

Reflection lets the agent assess whether the information it has is sufficient and consistent before acting. This is useful when data sources may be unreliable or contradictory — the agent can pause and ask itself: *"Do I trust this data enough to act on it?"*

In [ ]:
class ThoroughAnalyst(AmphibiousAutoma[CognitiveContext]):
    analyst = think_unit(
        CognitiveWorker.inline(
            "Analyze the stock using multiple data sources. "
            "Assess whether the data is consistent before making a recommendation.",
            enable_reflection=True,  # Enable information quality assessment
        ),
        max_attempts=8,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.analyst


agent = ThoroughAnalyst(llm=llm)
result = await agent.arun(
    goal="Analyze GOOGL stock — cross-reference financial data with sentiment before recommending",
    tools=[get_stock_price, get_financials, analyze_sentiment, generate_recommendation],
)
print(result)

With reflection enabled, the agent evaluates the quality and consistency of the data it has gathered before committing to a recommendation. If the financial metrics and sentiment data tell contradictory stories, the agent can flag this and adjust its recommendation accordingly.

Reflection is most useful when working with data from multiple sources that may not always agree.

## Combining Policies

All three policies can be enabled simultaneously. They fire in order: **Acquiring → Rehearsal → Reflection**. Each fires at most once per cycle. This gives your agent the maximum deliberation depth — it gathers information, simulates its plan, and then verifies data quality before committing to an action.

In [ ]:
class FullDeliberationAnalyst(AmphibiousAutoma[CognitiveContext]):
    analyst = think_unit(
        CognitiveWorker.inline(
            "Perform a comprehensive investment analysis. "
            "Gather all necessary information, simulate your recommendation, "
            "and verify data consistency before committing.",
            enable_rehearsal=True,
            enable_reflection=True,
            # Acquiring is always enabled by default
        ),
        max_attempts=10,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.analyst


agent = FullDeliberationAnalyst(llm=llm)
result = await agent.arun(
    goal="Provide a comprehensive investment analysis of MSFT for a conservative investor",
    tools=[get_stock_price, get_financials, analyze_sentiment, generate_recommendation],
)
print(result)

With all three policies active, the agent goes through the full deliberation pipeline within each OTC cycle:

1. **Acquiring** — the LLM requests detailed information from layered exposure fields if summaries are insufficient.
2. **Rehearsal** — the LLM mentally simulates its planned action to check for potential issues.
3. **Reflection** — the LLM assesses whether the collected data is consistent and reliable.

Only after all active policies have fired does the LLM commit to a final action.

> **Policy Rules:**
> - Each policy fires **at most once** per observe-think-act cycle, then closes
> - Policies fire in order: **Acquiring → Rehearsal → Reflection**
> - After all active policies have fired, the LLM must commit to a final action
> - Acquiring is built-in and always available; Rehearsal and Reflection must be explicitly enabled

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored the three cognitive policies that enhance agent deliberation:

- **Acquiring** (built-in, always active): Lets the LLM request details from `LayeredExposure` fields before deciding. The framework reveals requested information and re-triggers thinking.
- **Rehearsal** (`enable_rehearsal=True`): Lets the LLM mentally simulate its planned action before committing. Ideal for high-risk operations.
- **Reflection** (`enable_reflection=True`): Lets the LLM assess information quality and consistency before acting. Useful when data may be unreliable.
- All policies fire **at most once** per OTC cycle, in the order Acquiring → Rehearsal → Reflection.
- Policies can be combined — enable multiple policies on the same `CognitiveWorker` for maximum deliberation depth.

These policies add "thinking before acting" to your agent, improving decision quality for complex or high-stakes tasks.